# FootballAnalytics: volles Video auf Colab

Das Notebook führt Spieler-Tracking auf der GPU und die Frame-Registrierung auf der Colab-CPU aus. Das Eingabevideo und die fertigen Ergebnisse liegen dauerhaft in Google Drive; während der Rechnung wird lokal auf der schnellen Colab-SSD gearbeitet.

Vorher einmal in Google Drive anlegen: `Meine Ablage/FootballAnalytics/input/Video Project.mp4`.

In [ ]:
# Prüfen, ob Colab wirklich eine GPU zugeteilt hat.
import torch
assert torch.cuda.is_available(), 'Keine GPU aktiv: Laufzeit > Laufzeittyp ändern > GPU wählen'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Google Drive verbinden und Einstellungen festlegen.
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/FootballAnalytics')
VIDEO_NAME = 'Video Project.mp4'
REPO_URL = 'https://github.com/timg4/FootballAnalytics.git'
BRANCH = 'main'
STRIDE = 1       # 1 = beste Tracking-Qualität; 2 = schneller, aber mehr ID-Wechsel möglich
IMGSZ = 1280     # für die weit entfernten Spieler nicht kleiner stellen
MODEL = 'yolo11s.pt'
TRACKER = 'botsort-reid'  # mit Kamerakompensation + Appearance-ReID
RUN_LEGACY_REGISTRATION = False  # für die finale direkte Anker-Lokalisierung nicht mehr nötig

INPUT_VIDEO = DRIVE_ROOT / 'input' / VIDEO_NAME
RESULTS_DIR = DRIVE_ROOT / 'results' / Path(VIDEO_NAME).stem
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
assert INPUT_VIDEO.exists(), f'Video fehlt: {INPUT_VIDEO}'
print('Eingabe:', INPUT_VIDEO)
print('Ergebnisse:', RESULTS_DIR)

In [ ]:
# Code holen und Abhängigkeiten installieren.
import os, shutil, subprocess
WORK = Path('/content/FootballAnalytics')
if WORK.exists():
    shutil.rmtree(WORK)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORK)], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-r', str(WORK / 'requirements.txt')], check=True)
os.chdir(WORK)
print('Code-Version:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
# Das 1-GB-Video einmal auf die schnelle Laufzeit-SSD kopieren.
LOCAL_VIDEO = Path('/content') / VIDEO_NAME
if not LOCAL_VIDEO.exists() or LOCAL_VIDEO.stat().st_size != INPUT_VIDEO.stat().st_size:
    shutil.copy2(INPUT_VIDEO, LOCAL_VIDEO)
print(f'Lokale Kopie: {LOCAL_VIDEO} ({LOCAL_VIDEO.stat().st_size / 1e9:.2f} GB)')

In [ ]:
# Phase 1: YOLO + BoT-SORT/ReID auf der GPU. Vorhandenes ByteTrack-Ergebnis bleibt erhalten.
TRACKED_VIDEO = WORK / 'data/output/video_project_botsort_reid_tracked.mp4'
cmd = [
    'python', 'src/detect_track.py', str(LOCAL_VIDEO),
    '--model', MODEL, '--imgsz', str(IMGSZ), '--stride', str(STRIDE),
    '--device', '0', '--tracker', TRACKER, '--output', str(TRACKED_VIDEO),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Tracking sofort nach Drive sichern – so ist es auch bei einem späteren Runtime-Abbruch gerettet.
for path in (TRACKED_VIDEO, TRACKED_VIDEO.with_suffix('.csv')):
    target = RESULTS_DIR / path.name
    shutil.copy2(path, target)
    print('Gesichert:', target)

In [ ]:
# Optionaler Legacy-/Diagnoseschritt. Für Meterwerte wird inzwischen die driftfreie
# direkte Anker-Lokalisierung verwendet; deshalb standardmäßig überspringen.
HOMOGRAPHIES = WORK / 'data/output/video_project_homographies.npz'
if RUN_LEGACY_REGISTRATION:
    cmd = [
        'python', 'src/register_frames.py', str(LOCAL_VIDEO),
        '--ref', '0', '--stride', str(STRIDE), '--check',
        '--output', str(HOMOGRAPHIES),
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('Legacy-Registrierung übersprungen (korrekt für direkten Ankeransatz).')

In [ ]:
# Nur falls oben bewusst aktiviert: Legacy-Diagnoseartefakte sichern.
if RUN_LEGACY_REGISTRATION:
    artifacts = [HOMOGRAPHIES, *sorted((WORK / 'data/output').glob('check_warp_f*.jpg'))]
    for path in artifacts:
        target = RESULTS_DIR / path.name
        shutil.copy2(path, target)
        print('Gesichert:', target)
print('\nGPU-Teil fertig. Neue BoT-SORT/ReID-CSV lokal herunterladen und mit der vorhandenen Pitch-Lokalisierung auswerten.')